# NF fit of $h_{\rm dm}$, $\Delta t$ **and** $f_0$ — two-width $f_0$, complexity-penalised

Joint fit of the Hamiltonian-flow parameters ($h_{\rm dm}$, total time
$\Delta t = N_{\rm steps}\times dt$, $dt$ pinned at 0.5 Myr) **and the initial
distribution $f_0$**, on data whose true $f_0$ is not Gaussian:

$$f_0(z,w) = \mathcal N(z;0,\sigma_z)\,\big[0.6\,\mathcal N(w;0,15) + 0.4\,\mathcal N(w;0,60)\big]$$

(core + wings; N = 20000, seed 3 — the P2W dataset).  Both are results: the
fit must recover $(h_{\rm dm}, \Delta t)$ *and* the shape of $f_0$.

### Why a plain likelihood fit cannot do this
The flow is a 114-parameter RealNVP (3 couplings, hidden 4) — the smallest
stack that can make a non-Gaussian $w$-distribution.  But phase-space density
is conserved (Liouville), so for **any** $(h, \Delta t)$ there is an $f_0$ that
reproduces the observed snapshot exactly.  Measured: unpenalised, this flow
fits the stars *better* at the wrong teeth 437 and 348 Myr than at the true
400, and adding an EMD (pdf-vs-data) term does not change that.

### What does distinguish them: the shape of $f_0$
At the true $\Delta t$ the back-integrated cloud is the smooth core+wings
blob; at a wrong one it is a partly wound spiral with thin arms.  So the
objective penalises the **complexity of $f_0$**:

$$J(h,\Delta t;\lambda) = \min_\theta\Big[\,\mathrm{NLL}(\theta;h,\Delta t) + \lambda\,N\,R(\theta)\Big]$$

* **fisher** — $R$ = excess Fisher information of $f_0$,
  $\mathbb E|\nabla_y\log f_0|^2 - 2$ in standardised coordinates. Zero for a
  Gaussian, large for sharp features.  Measured at $h$=250: 2.2 at 400 Myr vs
  7.0 at 437 and 10.0 at 348.
* **l2** — weight decay $\sum\theta^2$, the simpler comparison.

$\lambda$ is **chosen by held-out likelihood**: fit on 4/5 of the stars, score
on the 1/5 the flow never saw, rotate (5-fold).  Too small a $\lambda$ memorises
the training stars; too large cannot make the core+wings shape.

### Pipeline (code: `experiments/penalized_f0.py`, each stage cached in `results/PF2w_*.pkl`)
| stage | what | est. time |
|---|---|---|
| 0 | candidate comb teeth from the cheap affine model — **proposals only** | ~4 min |
| 1 | 5-fold held-out NLL at every (tooth, penalty, $\lambda$) → picks penalty and $\lambda^*$ | ~15 min |
| 2 | **basin hopping from 8 initial guesses**, every local step a joint minimisation of all parameters | ~45–60 min |
| 3 | error bars from the full Hessian over all parameters | ~3 min |
| 4 | $f_0$ recovery: marginals, 2-D residual, held-out NLL vs the true $f_0$ | ~5 min |

In [ ]:
%matplotlib inline
import importlib, experiments.common, experiments.penalized_f0
importlib.reload(experiments.common); importlib.reload(experiments.penalized_f0)
import numpy as np
from experiments import common as C
from experiments import penalized_f0 as PF

N = 20_000
N_DISPLAY = 4 * N          # stars for the sim-data HEATMAP only -- NOT the fit
z_obs, w_obs = PF.load_data(N)
z_disp, w_disp = PF.load_data(N_DISPLAY)
true_pdf_z = lambda g: C.gauss_pdf(g, 0.0, C.Z0_SIGMA)
true_pdf_w = C.mixture_pdf(PF.COMPS, PF.WEIGHTS)

Q_CLIP = 97.0
zc = np.percentile(np.abs(z_obs - np.median(z_obs)), Q_CLIP)
wc = np.percentile(np.abs(w_obs - np.median(w_obs)), Q_CLIP)
ZLIM = (-zc, zc)
WLIM = (np.median(w_obs) - wc, np.median(w_obs) + wc)

nflow = sum(p.numel() for p in PF.make_flow(z_obs, w_obs).parameters())
print(f"N(fit)={N};  dt = {C.DT_STEP} Myr fixed;  flow = ZW({PF.FLOW_C},{PF.FLOW_HID}) "
      f"= {nflow} params;  truth h_dm = {C.H_TRUE:.0f} pc, Delta t = {C.DT_TRUE:.0f} Myr")

## Figure 1 — the simulated data

In [ ]:
C.plot_simulated_data(z_disp, w_disp, true_pdf_z, true_pdf_w,
                     zlim=ZLIM, wlim=WLIM,
                     title="Two-width $f_0$: $t=0$ and the $t=400$ Myr phase-space spiral");

---
## Stages 0–1 — candidate teeth, and choosing $\lambda$ by held-out likelihood

Stage 0 lists the deepest comb teeth of the affine model (cheap, closed form).
They are only *places to look*.  Stage 1 fits the 114-parameter flow at every
tooth for every penalty and $\lambda$, and scores it on held-out stars.

**Figure 2** — held-out NLL of each tooth vs $\lambda$.  Read it as: at
$\lambda=0$ the wrong teeth may win (the flow memorises their spiral); as
$\lambda$ grows, the **true tooth (thick line) should become the lowest**.  The
chosen $(\text{penalty}, \lambda^*)$ is the single lowest point on either panel.

In [ ]:
teeth = PF.candidate_teeth(z_obs, w_obs)
print("candidate teeth (affine proposals):")
for t in teeth["teeth"]:
    print(f"   Delta t = {t['dt']:5.0f} Myr   h = {t['h']:5.0f} pc   affine dNLL = {t['affine_dnll']:8.1f}")

scan = PF.lambda_scan(z_obs, w_obs, teeth["teeth"])
print(f"\nchosen: penalty = {scan['kind']},  lambda* = {scan['lam']:g},  "
      f"best tooth = {scan['tooth'][1]:.0f} Myr")
print("\nheld-out NLL minus the best [nats], rows = tooth:")
print(PF.lambda_table(scan).round(1))
PF.plot_lambda_scan(scan, title="Held-out likelihood vs penalty strength");

---
## Stages 2–3 — Figure 3: basin hopping from your initial guesses, every step a joint fit

The same algorithm as the Gaussian notebook (`local="joint"`), now with the
penalised 118-parameter model (4 affine + 114 coupling parameters):

1. start **at the initial guess** and minimise the penalised objective over
   **all parameters together** — $h_{\rm dm}$, $\Delta t$ and every flow
   weight — with L-BFGS on a 4000-star subsample;
2. jump in $(h_{\rm dm}, \Delta t)$ from the stored point, minimise jointly
   again, the flow starting from the stored point's weights;
3. keep the new point (and its flow) only if $J$ is lower; otherwise discard it;
4. stop after 15 jumps without improvement (at most 40);
5. joint minimisation of all parameters on the full 20000 stars.

8 starts spread over the box.  The background is the **affine** landscape —
there for orientation only; the penalised objective that drives the search has
no cheap landscape.  Error bars come from the full Hessian over all parameters
at the best landing (marginalised over the flow weights).

In [ ]:
hop = PF.hop_fit(z_obs, w_obs, scan)
err = PF.hop_errors(z_obs, w_obs, hop)
best = PF.best_run(hop)

import pandas as pd
pd.set_option("display.width", 200)
print("one row per initial guess (dJ_full = full-sample penalised objective minus the best):")
display(PF.hop_table(hop).round(3))

print(f"\nbest: h_dm    = {best['h_fit']:.1f} +/- {err['sig_h']:.1f} pc")
print(f"      Delta t = {best['dt_fit']:.2f} +/- {err['sig_dt']:.2f} Myr   (corr {err['corr']:+.3f}; "
      f"Hessian over all {err['n_params']} parameters, at a minimum: {err['is_min']})")
print(f"truth h_dm = {C.H_TRUE:.0f} pc, Delta t = {C.DT_TRUE:.0f} Myr")

C.plot_hdt_hop_paths(hop, n_paths=3, prof=dict(sig_h=err["sig_h"], sig_dt=err["sig_dt"]),
                     title="Two-width $f_0$: joint basin hopping, penalised 118-parameter flow");

---
## Figure 4 — the normalizing-flow fit of every run, one colour per run

Colours match the right panel of Figure 3; the legend gives each run's start,
landing and full-sample penalised objective relative to the best ($\Delta J$).

* **Top row — the fit to the DATA** ($t = 400$ Myr, observed coordinates).
  Grey: the observed stars.  Black: the true model (true $f_0$ carried to
  $t_{\rm obs}$ at the true $h_{\rm dm}, \Delta t$).  Colours: each run's model —
  its fitted $f_0$ carried to $t_{\rm obs}$ at *its own* $(h_{\rm dm}, \Delta t)$
  (exact: phase-space density is conserved, so $p_{\rm obs}(x) = f_0(T^{-1}x)$).
* **Bottom row — the recovered $f_0$** ($t = 0$).  Black: the true $f_0$ with
  its two generating $w$-Gaussians dotted; grey: the stars back-integrated at the
  TRUE $(h_{\rm dm}, \Delta t)$.

In [ ]:
PF.plot_run_fits(z_obs, w_obs, hop,
                 title="Two-width $f_0$: the normalizing-flow fit of every run");

---
## Stage 4 — Figure 5: how well is $f_0$ recovered?

The fitted $f_0$ at the final $(h_{\rm dm}, \Delta t)$, penalised ($\lambda^*$)
and unpenalised, against the true $f_0$ and its two generating Gaussians.
Bottom: fitted minus true $f_0$ at $t=0$.  The number in the legend is the
held-out NLL per star **minus that of the true $f_0$** on the same stars — 0
would be a perfect recovery, positive is worse than the truth.

In [ ]:
rep = PF.f0_report_hop(z_obs, w_obs, hop)
for tag in ("penalised", "unpenalised"):
    r = rep[tag]
    print(f"{tag:12s} lambda={r['lam']:<7g} held-out excess vs true f0 = "
          f"{r['excess_per_star']:+.4f} nats/star   excess Fisher R = {r['R']:.3f}")
print(f"{'true f0':12s} (on the cloud at the TRUE h, Delta t){'':22s}excess Fisher R = "
      f"{rep.get('R_true_on_true_cloud', rep['R_true']):.3f}")
print("held-out excess is measured against the TRUE MODEL of the observed stars (true f0 at the true h, Delta t)")
PF.plot_f0(z_obs, w_obs, rep,
           title=f"Recovered $f_0$ at $h$={rep['h']:.0f} pc, $\\Delta t$={rep['dt']:.1f} Myr");